# 04 · Transaction-level features

Build row-local signals: fee ratio, log amount, hour, weekday, weekend. No sender history yet.

In [ ]:
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

from cross_model_drift.data import load_transactions, write_parquet
from cross_model_drift.features import TRANSACTION_FEATURES, add_target, add_transaction_features
from cross_model_drift.notebook import setup_model_session

nb = setup_model_session()
v1_train = nb.split_plan().v1["train"]

raw = load_transactions(nb.config, engine=nb.engine, windows=[v1_train], limit=200_000)
feat = add_transaction_features(add_target(raw))

## Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.0))
sns.histplot(feat, x="log_amount", bins=40, ax=axes[0], color="#4C78A8")
axes[0].set_title("log1p(amount)")
sns.histplot(feat, x="fee_ratio", bins=40, ax=axes[1], color="#F58518")
axes[1].set_title("fee / amount")
sns.countplot(data=feat, x="hour_of_day", ax=axes[2], color="#72B7B2")
axes[2].set_title("hour of day")
nb.show(fig)

In [ ]:
by_hour = feat.groupby("hour_of_day", as_index=False)["is_fraud"].mean()
by_dow = feat.groupby(["day_of_week", "is_weekend"], as_index=False)["is_fraud"].mean()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
sns.barplot(data=by_hour, x="hour_of_day", y="is_fraud", ax=axes[0], color="#E45756")
axes[0].set_title("Fraud rate by hour")
axes[0].yaxis.set_major_formatter(lambda x, _: f"{x:.1%}")
sns.barplot(data=by_dow, x="day_of_week", y="is_fraud", hue="is_weekend", ax=axes[1])
axes[1].set_title("Fraud rate by weekday")
axes[1].yaxis.set_major_formatter(lambda x, _: f"{x:.1%}")
nb.show(fig)

## Persist a sample for later notebooks

In [ ]:
path = nb.artifacts / "samples" / "v1_train_txn_features.parquet"
write_parquet(feat, path)
path, len(feat)